In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # 在 notebook 里保存图片最稳
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
import numpy as np

import pandas as pd


# =========================================================
# 1. 你只需要改这里
# =========================================================
BASE_DIR = os.getenv("BEAVER_RUN_DIR", os.path.join("..", "Session_Runs", "YOUR_RUN_FOLDER"))
OUTPUT_DIR = os.path.join(BASE_DIR, "rebuild_plots")

# True = 直接覆盖原来的文件名
# False = 另存为 *_restyled.png
OVERWRITE_ORIGINAL_NAMES = False


# =========================================================
# 2. 全局风格参数
# =========================================================
FS_TITLE = 24
FS_LABEL = 22
FS_TICK = 18   
FS_ANNOT = 16
FS_LEGEND = 16



BAR_COLOR = (236/255, 183/255, 204/255)
ACCENT_COLOR = (120/255, 120/255, 120/255)
GRID_COLOR = "#E6E6E6"
RADAR_COLORS = [ (162/255, 204/255, 201/255), 
                 (234/255, 177/255, 200/255), 
                 (146/255, 190/255, 128/255)
               ]

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial"]


# =========================================================
# 3. 工具函数
# =========================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_float(x, default=0.0):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def style_spines(ax, lw=2.5):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_linewidth(lw)
    ax.spines["bottom"].set_linewidth(lw)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def try_load_records(path):
    try:
        if path.suffix.lower() == ".json":
            data = load_json(path)

            if isinstance(data, list) and all(isinstance(x, dict) for x in data):
                return data

            if isinstance(data, dict):
                for key in ["all_ideas", "top_ideas", "ideas", "results", "data", "records", "items"]:
                    val = data.get(key)
                    if isinstance(val, list) and all(isinstance(x, dict) for x in val):
                        return val

        elif path.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="gb18030")
            return df.to_dict(orient="records")

    except Exception:
        return None

    return None


def humanize_title_from_folder(folder_name):
    """
    idea1_Stereocomplex_Crystallites -> Stereocomplex Crystallites
    """
    name = re.sub(r"^idea\d+_", "", folder_name)
    name = name.replace("_", " ").strip()
    return name if name else folder_name


def sort_idea_dirs(idea_dirs):
    def extract_idx(p):
        m = re.match(r"idea(\d+)_", p.name)
        return int(m.group(1)) if m else 9999
    return sorted(idea_dirs, key=extract_idx)


# =========================================================
# 4. 初筛柱状图输入识别
# =========================================================
def score_field_exists(r):
    return (
        "score_overall" in r or
        "overall_score" in r or
        "score" in r
    )


def title_field_exists(r):
    return (
        "idea_name" in r or
        "title" in r or
        "idea" in r
    )


def normalize_initial_records(records):
    if not records:
        return None

    out = []
    for i, r in enumerate(records):
        if not isinstance(r, dict):
            return None

        if not title_field_exists(r) or not score_field_exists(r):
            return None

        name = r.get("idea_name", r.get("title", r.get("idea", f"Idea {i+1}")))
        score = r.get("score_overall", r.get("overall_score", r.get("score", 0)))

        out.append({
            "idea_name": str(name),
            "score_overall": safe_float(score, 0.0)
        })

    if len(out) < 2:
        return None

    return out


def find_root_initial_ideas_file(base_dir):
    candidates = []

    for p in Path(base_dir).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".csv"}:
            name_lower = p.name.lower()
            score = 100

            if "design_ideas" in name_lower and "top_ideas" not in name_lower:
                score = 0  # 给包含完整 8 个想法的初始文件最高优先级
            elif "all_ideas" in name_lower:
                score = 1
            elif "router" in name_lower:
                score = 2
            elif "design" in name_lower and "top_ideas" not in name_lower:
                score = 3
            elif "top_ideas" in name_lower:
                score = 4  # 降低 top_ideas 的优先级，防止它抢占柱状图的数据
            elif "ideas" in name_lower:
                score = 5

            skip_words = [
                "scores", "report_profile", "execution_context", "retrieval",
                "rerank", "paper", "report", "ragas", "reference", "evidence"
            ]
            if any(w in name_lower for w in skip_words):
                continue

            candidates.append((score, len(str(p)), p))

    candidates.sort(key=lambda x: (x[0], x[1]))

    for _, _, p in candidates:
        records = try_load_records(p)
        if records is None:
            continue
        normalized = normalize_initial_records(records)
        if normalized is not None:
            return p, normalized, "root_ideas_file"

    return None, None, "not_found"


# =========================================================
# 5. 从 idea 子目录重建雷达图输入
# =========================================================
def build_results_from_sidecars(base_dir):
    """
    返回：
    radar_results: [{title, scores_dict}, ...]
    bar_results:   [{idea_name, score_overall}, ...]
    """
    idea_dirs = [
        p for p in Path(base_dir).iterdir()
        if p.is_dir() and re.match(r"idea\d+_", p.name)
    ]
    idea_dirs = sort_idea_dirs(idea_dirs)

    radar_results = []
    bar_results = []

    for d in idea_dirs:
        score_files = list(d.glob("idea*_scores.json"))
        if not score_files:
            continue

        score_path = score_files[0]
        try:
            data = load_json(score_path)
        except Exception:
            continue

        if not isinstance(data, dict):
            continue

        title = humanize_title_from_folder(d.name)
        overall_score = safe_float(data.get("overall_score", 0), 0.0)
        scores_dict = data.get("scores_dict", {})

        if not isinstance(scores_dict, dict):
            scores_dict = {}

        normalized_scores = {
            "Feasibility": safe_float(scores_dict.get("Feasibility", 0), 0.0),
            "Predictability": safe_float(scores_dict.get("Predictability", 0), 0.0),
            "Performance": safe_float(scores_dict.get("Performance", 0), 0.0),
            "Innovation": safe_float(scores_dict.get("Innovation", 0), 0.0),
        }

        radar_results.append({
            "title": title,
            "scores_dict": normalized_scores
        })

        bar_results.append({
            "idea_name": title,
            "score_overall": overall_score
        })

    return radar_results, bar_results


# =========================================================
# 6. 画初筛柱状图
# =========================================================
def plot_initial_ranking(ideas, save_dir):
    names = []
    scores = []

    for idx, item in enumerate(ideas):
        raw_name = str(item.get("idea_name", f"Idea {idx+1}"))
        if len(raw_name) > 24:
            raw_name = raw_name[:21] + "..."
        names.append(raw_name)
        scores.append(safe_float(item.get("score_overall", 0), 0.0))

    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

    # === 1. 构建从深到浅的粉色渐变带 ===
    # 最高分用深一点的玫瑰色，中间分用你选的温柔粉，最低分用极浅的粉白
    dark_pink  = (190/255, 100/255, 140/255)   # 深玫瑰色 (对应高分)
    base_pink  = (236/255, 183/255, 204/255)   # 你的指定色 (对应中间分)
    light_pink = (252/255, 238/255, 242/255)   # 浅粉白色 (对应低分)
    
    # 创建渐变映射
    cmap = LinearSegmentedColormap.from_list("custom_pink", [light_pink, base_pink, dark_pink])

    # === 2. 将分数映射到颜色 ===
    # 我们把 vmin 稍微设得比最低分小一点，防止最低分的柱子变成纯白色看不见
    norm = Normalize(vmin=min(scores) - 1.0, vmax=max(scores)) 
    bar_colors = [cmap(norm(s)) for s in scores]

    # === 3. 画柱状图 ===
    bars = ax.bar(
        names,
        scores,
        color=bar_colors,                       # <--- 使用计算好的渐变色列表
        edgecolor="black",  # <--- 搭配一个高级的深玫瑰灰边框
        linewidth=1.75,
        alpha=1.00,
        width=0.75,
        zorder=3
    )

    if len(scores) >= 3:
        cutoff = sorted(scores, reverse=True)[2]
        ax.axhline(
            y=cutoff,
            color=ACCENT_COLOR,
            linestyle="--",
            linewidth=2.2,
            alpha=0.95,
            zorder=2
        )
        ax.text(
            len(names) - 0.3,
            cutoff + 0.12,
            f"Selection Threshold ({cutoff:.2f})",
            color=ACCENT_COLOR,
            ha="right",
            va="bottom",
            fontsize=FS_ANNOT,
            fontweight="bold"
        )

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.08,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=FS_ANNOT,
            fontweight="bold",
            color="black"
        )

    ax.set_ylabel("Overall Score (0-10)", fontsize=FS_LABEL, fontweight="bold")
    ax.set_title(
        f"Idea Divergence Analysis: Top {len(ideas)} Candidates",
        fontsize=FS_TITLE,
        fontweight="bold",
        pad=12
    )

    ax.tick_params(axis="x", labelsize=FS_TICK)
    ax.tick_params(axis="y", labelsize=FS_TICK)

    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_rotation_mode("anchor")
        label.set_ha("right")
        label.set_fontweight("bold")
        label.set_color("black")

    plt.subplots_adjust(bottom=0.24)

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")
        label.set_color("black")

    ax.grid(axis="y", linestyle="--", alpha=0.35, color=GRID_COLOR, zorder=0)
    style_spines(ax, lw=2.5)
    ax.set_ylim(0, max(11, max(scores) + 1.0))

    plt.tight_layout()

    if OVERWRITE_ORIGINAL_NAMES:
        out_path = os.path.join(save_dir, "initial_screening_bar.png")
    else:
        out_path = os.path.join(save_dir, "initial_screening_bar_restyled.png")

    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"[BAR] Saved: {out_path}")
    return out_path


# =========================================================
# 7. 画雷达图
# =========================================================
def plot_final_radar(results, save_dir):
    categories = ["Feasibility", "Predictability", "Performance", "Innovation"]
    display_labels = ["Feasibility", "Predictability", "Performance", "Innovation"]

    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles_closed = angles + angles[:1]

    fig, ax = plt.subplots(
        figsize=(9.2, 9.2),
        subplot_kw=dict(polar=True),
        dpi=300
    )
    fig.subplots_adjust(top=0.84, bottom=0.20, left=0.10, right=0.90)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.set_ylim(0, 100)

    # ===== 1) 不再用默认 xticks 直接贴类别标签 =====
    ax.set_xticks(angles)
    ax.set_xticklabels([])

    # ===== 2) 把半径刻度挪开，不放在容易被线挡住的位置 =====
    # ===== 手工控制径向刻度文字 =====
    ax.set_rlabel_position(135)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels([])   # 先把默认的隐藏
    
    tick_theta = np.deg2rad(132)   # 刻度整体放在哪个角度，可微调 128~138
    tick_style = dict(
        fontsize=FS_ANNOT,
        fontweight="bold",
        color="black",
        zorder=30,
        bbox=dict(
            boxstyle="round,pad=0.16",
            facecolor="white",
            edgecolor="none",
            alpha=0.96
        )
    )
    
    # 半径位置单独微调，避免 20 太靠里、100 太贴边
    manual_radii = {
        20: 18.5,
        40: 39.0,
        60: 59.0,
        80: 79.0,
        100: 96.5
    }
    
    for val, rpos in manual_radii.items():
        ax.text(
            tick_theta,
            rpos,
            f"{val}",
            ha="center",
            va="center",
            **tick_style
        )

    # ===== 3) 手工固定四个指标名的位置（用画布坐标，不再用极坐标半径放） =====
    label_kw = dict(
        fontsize=FS_TICK,
        fontweight="bold",
        color="black",
        clip_on=False,
        zorder=40,
        bbox=dict(
            boxstyle="round,pad=0.22",
            facecolor="white",
            edgecolor="none",
            alpha=0.96
        )
    )
    
    ax.text(
        0.50, 1.03, "Feasibility",
        transform=ax.transAxes,
        ha="center", va="bottom",
        **label_kw
    )
    
    ax.text(
        1.03, 0.50, "Predictability",
        transform=ax.transAxes,
        ha="left", va="center",
        **label_kw
    )
    
    ax.text(
        0.50, -0.03, "Performance",
        transform=ax.transAxes,
        ha="center", va="top",
        **label_kw
    )
    
    ax.text(
        -0.03, 0.50, "Innovation",
        transform=ax.transAxes,
        ha="right", va="center",
        **label_kw
    )

    # ===== 4) 画多条雷达线 =====
    for idx, res in enumerate(results):
        s_dict = res.get("scores_dict", {})
        values = [safe_float(s_dict.get(c, 0), 0.0) for c in categories]
        values += values[:1]

        color = RADAR_COLORS[idx % len(RADAR_COLORS)]
        label_text = str(res.get("title", f"Idea {idx+1}"))
        if len(label_text) > 28:
            label_text = label_text[:25] + "..."

        ax.plot(
            angles_closed,
            values,
            linewidth=2.2,
            linestyle="-",
            label=label_text,
            color=color,
            alpha=0.90,
            zorder=4
        )
        ax.fill(
            angles_closed,
            values,
            color=color,
            alpha=0.10,
            zorder=2
        )

    ax.grid(color=GRID_COLOR, linestyle="--", linewidth=1.1)
    ax.spines["polar"].set_color("black")
    ax.spines["polar"].set_linewidth(2.2)

    plt.title(
        "Convergence: Multi-Dimensional Evaluation",
        size=20,
        y=1.12,
        fontweight="bold",
        color="black"
    )

    plt.legend(
        loc="lower right",
        bbox_to_anchor=(1.40, -0.15),
        ncol=1,
        frameon=False,
        fontsize=FS_LEGEND
    )

    if OVERWRITE_ORIGINAL_NAMES:
        out_path = os.path.join(save_dir, "final_verification_radar.png")
    else:
        out_path = os.path.join(save_dir, "final_verification_radar_restyled.png")

    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"[RADAR] Saved: {out_path}")
    return out_path

# =========================================================
# 8. 主函数
# =========================================================
def rebuild_plots(base_dir=BASE_DIR, output_dir=OUTPUT_DIR):
    ensure_dir(output_dir)

    manifest = {
        "base_dir": base_dir,
        "initial_bar": {
            "source_type": None,
            "source_file": None,
            "n_records": 0,
            "output_png": None,
            "status": "not_run"
        },
        "radar": {
            "source_type": None,
            "source_file": None,
            "n_records": 0,
            "output_png": None,
            "status": "not_run"
        }
    }

    print("=" * 90)
    print("开始处理目录：")
    print(base_dir)
    print("=" * 90)

    # 1) root 下找初筛 ideas
    initial_file, initial_data, initial_source_type = find_root_initial_ideas_file(base_dir)

    # 2) idea 子目录读 scores
    radar_data, fallback_bar_data = build_results_from_sidecars(base_dir)

    # -------- BAR --------
    if initial_data:
        print(f"[BAR] 使用 root ideas 文件：{initial_file}")
        out_bar = plot_initial_ranking(initial_data, output_dir)

        manifest["initial_bar"]["source_type"] = initial_source_type
        manifest["initial_bar"]["source_file"] = str(initial_file)
        manifest["initial_bar"]["n_records"] = len(initial_data)
        manifest["initial_bar"]["output_png"] = out_bar
        manifest["initial_bar"]["status"] = "ok"

    elif fallback_bar_data:
        print("[BAR] 未找到 root 初筛 ideas 文件，退化为使用各个 idea 子目录的 overall_score 重建")
        out_bar = plot_initial_ranking(fallback_bar_data, output_dir)

        manifest["initial_bar"]["source_type"] = "idea_score_sidecars_fallback"
        manifest["initial_bar"]["source_file"] = "idea*/idea*_scores.json"
        manifest["initial_bar"]["n_records"] = len(fallback_bar_data)
        manifest["initial_bar"]["output_png"] = out_bar
        manifest["initial_bar"]["status"] = "ok_fallback"

    else:
        print("[BAR] 没找到可用输入")
        manifest["initial_bar"]["status"] = "missing_input"

    # -------- RADAR --------
    if radar_data:
        print("[RADAR] 使用各个 idea 子目录的 idea*_scores.json 重建")
        out_radar = plot_final_radar(radar_data, output_dir)

        manifest["radar"]["source_type"] = "idea_score_sidecars"
        manifest["radar"]["source_file"] = "idea*/idea*_scores.json"
        manifest["radar"]["n_records"] = len(radar_data)
        manifest["radar"]["output_png"] = out_radar
        manifest["radar"]["status"] = "ok"

    else:
        print("[RADAR] 没找到可用输入")
        manifest["radar"]["status"] = "missing_input"

    # manifest
    manifest_path = os.path.join(output_dir, "rebuild_plots_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"\nManifest 已保存：{manifest_path}")
    print("完成。只会输出两张图。")

    return manifest


# =========================================================
# 9. 最后调用（你在 .ipynb 里直接运行到这里就行）
# =========================================================
manifest = rebuild_plots(BASE_DIR, OUTPUT_DIR)
manifest

In [ ]:
# -*- coding: utf-8 -*-
"""
纯调用版 wrapper：
- 不重写 plot 逻辑
- 不在本脚本里重新实现画图函数
- 只负责：找输入数据 -> 导入 idea_visualizer.py -> 直接调用其中的 plot_initial_ranking / plot_final_radar

你只需要改最上面的 3 个路径：
1) BASE_DIR
2) OUTPUT_DIR
3) IDEA_VISUALIZER_PATH
"""

import os
import re
import sys
import json
import importlib.util
from pathlib import Path

import pandas as pd


# =========================================================
# 1. 只改这里
# =========================================================
BASE_DIR = os.getenv("BEAVER_RUN_DIR", os.path.join("..", "Session_Runs", "YOUR_RUN_FOLDER"))
OUTPUT_DIR = os.path.join(BASE_DIR, "rebuild_plots")
IDEA_VISUALIZER_PATH = os.getenv("BEAVER_IDEA_VISUALIZER", "idea_visualizer.py")


# =========================================================
# 2. 通用工具
# =========================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def safe_float(x, default=0.0):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def try_load_records(path: Path):
    try:
        if path.suffix.lower() == ".json":
            data = load_json(path)
            if isinstance(data, list) and all(isinstance(x, dict) for x in data):
                return data
            if isinstance(data, dict):
                for key in ["all_ideas", "top_ideas", "ideas", "results", "data", "records", "items"]:
                    val = data.get(key)
                    if isinstance(val, list) and all(isinstance(x, dict) for x in val):
                        return val
        elif path.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="gb18030")
            return df.to_dict(orient="records")
    except Exception:
        return None
    return None


def score_field_exists(r):
    return (
        "score_overall" in r or
        "overall_score" in r or
        "score" in r
    )


def title_field_exists(r):
    return (
        "idea_name" in r or
        "title" in r or
        "idea" in r
    )


def normalize_initial_records(records):
    if not records:
        return None

    out = []
    for i, r in enumerate(records):
        if not isinstance(r, dict):
            return None
        if not title_field_exists(r) or not score_field_exists(r):
            return None

        name = r.get("idea_name", r.get("title", r.get("idea", f"Idea {i+1}")))
        score = r.get("score_overall", r.get("overall_score", r.get("score", 0)))

        out.append({
            "idea_name": str(name),
            "score_overall": safe_float(score, 0.0)
        })

    if len(out) < 2:
        return None
    return out


def find_root_initial_ideas_file(base_dir):
    candidates = []

    for p in Path(base_dir).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".json", ".csv"}:
            name_lower = p.name.lower()
            score = 100

            if "design_ideas" in name_lower and "top_ideas" not in name_lower:
                score = 0
            elif "all_ideas" in name_lower:
                score = 1
            elif "router" in name_lower:
                score = 2
            elif "design" in name_lower and "top_ideas" not in name_lower:
                score = 3
            elif "top_ideas" in name_lower:
                score = 4
            elif "ideas" in name_lower:
                score = 5

            skip_words = [
                "scores", "report_profile", "execution_context", "retrieval",
                "rerank", "paper", "report", "ragas", "reference", "evidence"
            ]
            if any(w in name_lower for w in skip_words):
                continue

            candidates.append((score, len(str(p)), p))

    candidates.sort(key=lambda x: (x[0], x[1]))

    for _, _, p in candidates:
        records = try_load_records(p)
        if records is None:
            continue
        normalized = normalize_initial_records(records)
        if normalized is not None:
            return p, normalized, "root_ideas_file"

    return None, None, "not_found"


def humanize_title_from_folder(folder_name):
    name = re.sub(r"^idea\d+_", "", folder_name)
    name = name.replace("_", " ").strip()
    return name if name else folder_name


def sort_idea_dirs(idea_dirs):
    def extract_idx(p):
        m = re.match(r"idea(\d+)_", p.name)
        return int(m.group(1)) if m else 9999
    return sorted(idea_dirs, key=extract_idx)


def build_results_from_sidecars(base_dir):
    idea_dirs = [
        p for p in Path(base_dir).iterdir()
        if p.is_dir() and re.match(r"idea\d+_", p.name)
    ]
    idea_dirs = sort_idea_dirs(idea_dirs)

    radar_results = []
    bar_results = []

    for d in idea_dirs:
        score_files = list(d.glob("idea*_scores.json"))
        if not score_files:
            continue

        score_path = score_files[0]
        try:
            data = load_json(score_path)
        except Exception:
            continue

        if not isinstance(data, dict):
            continue

        title = humanize_title_from_folder(d.name)
        overall_score = safe_float(data.get("overall_score", 0), 0.0)
        scores_dict = data.get("scores_dict", {})
        if not isinstance(scores_dict, dict):
            scores_dict = {}

        normalized_scores = {
            "Feasibility": safe_float(scores_dict.get("Feasibility", 0), 0.0),
            "Predictability": safe_float(scores_dict.get("Predictability", 0), 0.0),
            "Performance": safe_float(scores_dict.get("Performance", 0), 0.0),
            "Innovation": safe_float(scores_dict.get("Innovation", 0), 0.0),
        }

        radar_results.append({
            "title": title,
            "scores_dict": normalized_scores
        })

        bar_results.append({
            "idea_name": title,
            "score_overall": overall_score
        })

    return radar_results, bar_results


# =========================================================
# 3. 动态导入你的 idea_visualizer.py
# =========================================================
def import_idea_visualizer(module_path: str):
    module_path = os.path.abspath(module_path)
    if not os.path.exists(module_path):
        raise FileNotFoundError(f"找不到 idea_visualizer.py: {module_path}")

    spec = importlib.util.spec_from_file_location("idea_visualizer_user", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"无法加载模块: {module_path}")

    module = importlib.util.module_from_spec(spec)
    sys.modules["idea_visualizer_user"] = module
    spec.loader.exec_module(module)

    if not hasattr(module, "plot_initial_ranking"):
        raise AttributeError("idea_visualizer.py 缺少函数: plot_initial_ranking")
    if not hasattr(module, "plot_final_radar"):
        raise AttributeError("idea_visualizer.py 缺少函数: plot_final_radar")

    return module


# =========================================================
# 4. 主函数：只调用，不重写画图
# =========================================================
def main(base_dir=BASE_DIR, output_dir=OUTPUT_DIR, idea_visualizer_path=IDEA_VISUALIZER_PATH):
    ensure_dir(output_dir)

    vis = import_idea_visualizer(idea_visualizer_path)

    manifest = {
        "base_dir": base_dir,
        "output_dir": output_dir,
        "idea_visualizer_path": idea_visualizer_path,
        "initial_bar": {
            "source_type": None,
            "source_file": None,
            "n_records": 0,
            "status": "not_run"
        },
        "radar": {
            "source_type": None,
            "source_file": None,
            "n_records": 0,
            "status": "not_run"
        }
    }

    print("=" * 90)
    print("开始调用 idea_visualizer.py")
    print(f"BASE_DIR   : {base_dir}")
    print(f"OUTPUT_DIR : {output_dir}")
    print(f"VISUALIZER : {idea_visualizer_path}")
    print("=" * 90)

    # 1) root 下找初筛 ideas 文件
    initial_file, initial_data, initial_source_type = find_root_initial_ideas_file(base_dir)

    # 2) 从 idea 子目录重建雷达图输入
    radar_data, fallback_bar_data = build_results_from_sidecars(base_dir)

    # -------- 初筛柱状图：直接调用 idea_visualizer.py --------
    if initial_data:
        print(f"[BAR] 使用 root ideas 文件: {initial_file}")
        vis.plot_initial_ranking(initial_data, output_dir)
        manifest["initial_bar"].update({
            "source_type": initial_source_type,
            "source_file": str(initial_file),
            "n_records": len(initial_data),
            "status": "ok"
        })
    elif fallback_bar_data:
        print("[BAR] 未找到 root ideas 文件，退化为使用 idea*_scores.json 的 overall_score")
        vis.plot_initial_ranking(fallback_bar_data, output_dir)
        manifest["initial_bar"].update({
            "source_type": "idea_score_sidecars_fallback",
            "source_file": "idea*/idea*_scores.json",
            "n_records": len(fallback_bar_data),
            "status": "ok_fallback"
        })
    else:
        print("[BAR] 没找到可用输入，跳过")
        manifest["initial_bar"]["status"] = "missing_input"

    # -------- 雷达图：直接调用 idea_visualizer.py --------
    if radar_data:
        print("[RADAR] 使用 idea*/idea*_scores.json")
        vis.plot_final_radar(radar_data, output_dir)
        manifest["radar"].update({
            "source_type": "idea_score_sidecars",
            "source_file": "idea*/idea*_scores.json",
            "n_records": len(radar_data),
            "status": "ok"
        })
    else:
        print("[RADAR] 没找到可用输入，跳过")
        manifest["radar"]["status"] = "missing_input"

    manifest_path = os.path.join(output_dir, "idea_visualizer_wrapper_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"\nManifest 已保存: {manifest_path}")
    print("完成。这个 wrapper 只负责调用 idea_visualizer.py，不在本脚本里重写画图函数。")


if __name__ == "__main__":
    main()
